In [ ]:
import time
from pynput.mouse import Controller

mouse = Controller()

print("Posicione o mouse... você tem 5 segundos")
time.sleep(5)

x, y = mouse.position
print(f"Posição do mouse: X={x}, Y={y}")

In [ ]:
import re
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import quote_plus

BASE_URL = "https://www.transfermarkt.com"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}

SESSION = requests.Session()
SESSION.headers.update(HEADERS)

def _sleep():
    time.sleep(1.2)

def search_player_profile_url(player_name: str) -> str | None:
    """
    Busca o jogador pela schnellsuche e retorna a URL do perfil.
    """
    q = quote_plus(player_name.strip())
    url = f"{BASE_URL}/schnellsuche/ergebnis/schnellsuche?query={q}"

    r = SESSION.get(url, timeout=12)
    _sleep()
    if r.status_code != 200:
        return None

    soup = BeautifulSoup(r.text, "lxml")

    # Primeiro resultado de jogador (geralmente em table.items)
    a = soup.select_one("table.items tbody tr td.hauptlink a")
    if not a or not a.get("href"):
        return None

    href = a["href"].strip()
    if href.startswith("http"):
        return href
    return BASE_URL + href

def _to_rumours_url(profile_url: str) -> str:
    """
    Converte /profil/ para /geruechte/ mantendo o restante.
    Ex: /endrick/profil/spieler/971570 -> /endrick/geruechte/spieler/971570
    """
    return re.sub(r"/profil/", "/geruechte/", profile_url)

def get_rumours_from_profile(profile_url: str) -> pd.DataFrame:
    """
    Extrai a tabela Rumours do Transfermarkt no formato do HTML que você colou.
    """
    rumours_url = _to_rumours_url(profile_url)

    r = SESSION.get(rumours_url, timeout=12)
    _sleep()
    if r.status_code != 200:
        return pd.DataFrame()

    soup = BeautifulSoup(r.text, "lxml")

    # A tabela que você mostrou: <table class="items">
    table = soup.select_one("div.box table.items")
    if not table:
        return pd.DataFrame()

    out = []
    for tr in table.select("tbody tr"):
        tds = tr.find_all("td")
        # Estrutura típica do Rumours:
        # 0 = logo clube
        # 1 = nome clube (com link)
        # 2 = Most recent source (link com data)
        # 3 = Last reply (link com data)
        # 4 = Probability (texto "72 %", + link thread)
        if len(tds) < 5:
            continue

        # Clube interessado
        club_a = tds[1].select_one("a")
        interested_club = club_a.get_text(strip=True) if club_a else tds[1].get_text(strip=True)
        club_url = ""
        if club_a and club_a.get("href"):
            club_url = club_a["href"].strip()
            if club_url and not club_url.startswith("http"):
                club_url = BASE_URL + club_url

        # Most recent source
        src_a = tds[2].select_one("a")
        most_recent_source_date = src_a.get_text(strip=True) if src_a else tds[2].get_text(strip=True)
        most_recent_source_url = ""
        thread_title_src = ""
        if src_a and src_a.get("href"):
            most_recent_source_url = src_a["href"].strip()
            if most_recent_source_url and not most_recent_source_url.startswith("http"):
                most_recent_source_url = BASE_URL + most_recent_source_url
            thread_title_src = (src_a.get("title") or "").strip()

        # Last reply
        reply_a = tds[3].select_one("a")
        last_reply_date = reply_a.get_text(strip=True) if reply_a else tds[3].get_text(strip=True)
        last_reply_url = ""
        thread_title_reply = ""
        if reply_a and reply_a.get("href"):
            last_reply_url = reply_a["href"].strip()
            if last_reply_url and not last_reply_url.startswith("http"):
                last_reply_url = BASE_URL + last_reply_url
            thread_title_reply = (reply_a.get("title") or "").strip()

        # Probability
        prob_text = tds[4].get_text(" ", strip=True)  # ex: "72 %"
        m = re.search(r"(\d+)\s*%", prob_text)
        probability_percent = int(m.group(1)) if m else None

        # Às vezes o link do thread está dentro desta célula também
        # Pegamos o primeiro <a> que pareça ser do thread (geralmente com title)
        prob_thread_a = tds[4].select_one("a[title][href]")
        probability_thread_url = ""
        thread_title_prob = ""
        if prob_thread_a and prob_thread_a.get("href"):
            probability_thread_url = prob_thread_a["href"].strip()
            if probability_thread_url and not probability_thread_url.startswith("http"):
                probability_thread_url = BASE_URL + probability_thread_url
            thread_title_prob = (prob_thread_a.get("title") or "").strip()

        # Escolhe o melhor título disponível
        thread_title = thread_title_prob or thread_title_reply or thread_title_src

        out.append({
            "interested_club": interested_club,
            "club_url": club_url,
            "most_recent_source_date": most_recent_source_date,
            "most_recent_source_url": most_recent_source_url,
            "last_reply_date": last_reply_date,
            "last_reply_url": last_reply_url,
            "probability_percent": probability_percent,
            "thread_title": thread_title,
            "rumours_url": rumours_url,
        })

    return pd.DataFrame(out)

if __name__ == "__main__":
    name = input("Nome do jogador: ").strip()

    profile = search_player_profile_url(name)
    if not profile:
        print("❌ Jogador não encontrado na busca.")
        raise SystemExit(1)

    print("✅ Perfil:", profile)

    df = get_rumours_from_profile(profile)
    if df.empty:
        print("⚠️ Rumours: tabela vazia (pode ser que não haja rumores abertos).")
    else:
        print(df.to_string(index=False))


In [ ]:
import os
import requests
import urllib.parse
from dotenv import load_dotenv

# =========================================================
# Carrega .env
# =========================================================
load_dotenv()

YOUTUBE_API_KEY = os.getenv("YOUTUBE_API_KEY")

if not YOUTUBE_API_KEY:
    raise RuntimeError("YOUTUBE_API_KEY não encontrada no .env")

# =========================================================
# Função principal: busca highlights no YouTube
# =========================================================
def search_youtube_highlights(
    player_name: str,
    team_name: str | None = None,
    max_results: int = 6,
):
    """
    Retorna lista de vídeos do YouTube relacionados a highlights do jogador.

    Output:
    [
      {
        "videoId": "...",
        "title": "...",
        "channel": "...",
        "publishedAt": "...",
        "url": "https://www.youtube.com/watch?v=..."
      }
    ]
    """

    query_parts = [player_name, "highlights"]
    if team_name:
        query_parts.append(team_name)

    query = " ".join(query_parts)

    params = {
        "part": "snippet",
        "q": query,
        "type": "video",
        "videoDuration": "medium",   # evita vídeos muito curtos
        "order": "relevance",
        "maxResults": max_results,
        "safeSearch": "strict",
        "key": YOUTUBE_API_KEY,
    }

    url = "https://www.googleapis.com/youtube/v3/search"

    try:
        resp = requests.get(url, params=params, timeout=8)
        resp.raise_for_status()
        data = resp.json()
    except Exception as e:
        print("❌ Erro ao chamar API do YouTube:", e)
        return []

    items = []

    for it in data.get("items", []):
        try:
            vid = it["id"]["videoId"]
            sn = it["snippet"]
            items.append(
                {
                    "videoId": vid,
                    "title": sn.get("title"),
                    "channel": sn.get("channelTitle"),
                    "publishedAt": sn.get("publishedAt"),
                    "url": f"https://www.youtube.com/watch?v={vid}",
                }
            )
        except Exception:
            continue

    return items


# =========================================================
# TESTE MANUAL
# =========================================================
if __name__ == "__main__":
    player = "Yan Diomande"
    team = "RB Leipzig"

    results = search_youtube_highlights(player, team)

    print(f"\n🎥 Vídeos encontrados para {player}:\n")
    for i in results:
        print(f"- {i['title']} ({i['channel']}, {i['publishedAt']})")
        print(f"  {i['url']}\n")

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("stats.csv")
display(df["leaguename"].unique())

array(['Premier League', 'LaLiga', 'Serie A', 'Bundesliga', 'Ligue 1',
       'Brasileirão Betano', 'Saudi Pro League', 'Liga Portugal Betclic',
       'VriendenLoterij Eredivisie', 'Championship', 'Trendyol Süper Lig',
       'MLS', 'Pro League', 'Egyptian Premier League',
       'Liga Profesional de Fútbol', '2. Bundesliga', 'LaLiga 2',
       'Scottish Premiership', 'Stoiximan Super League', 'HNL',
       'Liga MX, Apertura', 'Liga MX, Clausura', 'Serie B',
       'Danish Superliga', 'Swiss Super League', 'Ligue 2',
       'Russian Premier League', 'Ekstraklasa', 'Eliteserien',
       'Allsvenskan', 'League One', 'Eerste Divisie',
       'Austrian Bundesliga', 'Botola Pro', 'Primera A, Apertura',
       'A-League Men', 'J1 League', 'Liga 1', 'Primera A, Finalización',
       'Romanian SuperLiga', 'South African Premier Division',
       'Liga de Primera', 'Czech First League', 'Indonesia Super League',
       'Mozzart Bet Superliga', 'Chinese Super League', 'LigaPro Serie A',
      

In [3]:
import findspark
from pyspark.sql.functions import *
findspark.init()

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("antt_pipeline")
    .master("local[*]")  # usa todos os cores da máquina
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "8g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.caseSensitive", "true")

def display_spark(df,limit = None):
    try:
        if limit:
            pdf = df.limit(limit).toPandas()
            display(pdf)
        else:
            pdf = df.toPandas()
            display(pdf)
    except Exception as e:
        print("Erro ao converter para Pandas:", e)
        print("Exibindo Spark DataFrame diretamente:")
        df.show(10, truncate=False)

In [4]:
df = spark.read.csv("stats.csv", header=True, inferSchema=True)

In [5]:
for column in df.columns:
    print(f"{column}: {df.schema[column].dataType}")

playerid: IntegerType()
playername: StringType()
position: StringType()
teamid: IntegerType()
teamname: StringType()
dateofbirthtimestamp: DoubleType()
age: DoubleType()
positionsdetailed: StringType()
height: DoubleType()
preferredfoot: StringType()
playercountry: StringType()
proposedmarketvalue: DoubleType()
contractuntil: StringType()
tournamentid: IntegerType()
seasonid: IntegerType()
accuratecrosses: DoubleType()
accuratecrossespercentage: DoubleType()
accuratelongballs: DoubleType()
accuratelongballspercentage: DoubleType()
accuratepasses: DoubleType()
accuratepassespercentage: DoubleType()
aerialduelswon: DoubleType()
assists: IntegerType()
bigchancescreated: IntegerType()
bigchancesmissed: IntegerType()
blockedshots: DoubleType()
outfielderblocks: DoubleType()
cleansheet: IntegerType()
dribbledpast: IntegerType()
errorleadtogoal: IntegerType()
expectedassists: DoubleType()
expectedgoals: DoubleType()
goals: IntegerType()
goalsassistssum: IntegerType()
goalsconceded: IntegerTyp

In [6]:
ligas = df.groupBy("leaguename").agg(avg("usercount").alias("league_reputation")).orderBy(col("league_reputation").desc())
lista_ligas = [row["leaguename"] for row in ligas.filter((col("league_reputation") > 20000) & (col("leaguename")!= 'League One')).collect()]

In [7]:
lista_ligas

['Premier League',
 'LaLiga',
 'Serie A',
 'Bundesliga',
 'Ligue 1',
 'Brasileirão Betano',
 'Saudi Pro League',
 'Liga Portugal Betclic',
 'VriendenLoterij Eredivisie',
 'Championship',
 'Trendyol Süper Lig',
 'MLS',
 'Pro League',
 'Egyptian Premier League',
 'Liga Profesional de Fútbol',
 '2. Bundesliga',
 'LaLiga 2',
 'Scottish Premiership',
 'Stoiximan Super League',
 'HNL',
 'Liga MX, Apertura',
 'Liga MX, Clausura',
 'Serie B',
 'Danish Superliga',
 'Swiss Super League',
 'Ligue 2',
 'Russian Premier League',
 'Ekstraklasa',
 'Eliteserien',
 'Allsvenskan',
 'Eerste Divisie',
 'Austrian Bundesliga',
 'Botola Pro',
 'Primera A, Apertura',
 'A-League Men',
 'J1 League']

In [8]:
display_spark(df.filter((col("rating") > 7) & (col("minutesplayed") >800)&(col("age") <= 24) & (col("leaguename").isin(lista_ligas))).groupBy("playercountry").agg(count("*").alias("count"),collect_set("playername").alias("players")).orderBy(col("count").desc()))

,playercountry,count,players
0,Netherlands,49,"[Niek Schiks, Illaijh de Ruijter, Finn Stam, S..."
1,France,31,"[Manu Koné, Yllan Okou, Kevin Pedro, Castello ..."
2,Sweden,30,"[Samuel Dahl, Matteo Pérez Vinlöf, Otto Roseng..."
3,Brazil,29,"[João Peglow, Jhon Jhon, Bernardo Fontes, Wesl..."
4,Spain,28,"[Lamine Yamal, Aitor Fraga, Pedri, Pau Cubarsí..."
...,...,...,...
71,Malta,1,[Dylan Scicluna]
72,Suriname,1,[Melayro Bogarde]
73,Palestine,1,[Omar Faraj]
74,Pakistan,1,[Abdullah Iqbal]
